# Feature Engineering Assignment – HR Attrition Dataset

**Objective:** Build a defensible, reusable feature-engineering pipeline for an enterprise-style HR attrition dataset.

This notebook covers:
- Part A: dataset understanding and exploratory analysis
- Part B: missing-value handling, transformations and encoding
- Part C: feature construction and feature selection
- Part D: dimensionality reduction and reusable pipeline design
- Part E: summary, assumptions and limitations

> The target variable is **Attrition** (0 = Stayed, 1 = Left). No predictive model is trained because model accuracy is explicitly outside the assignment scope.


## 1. Business Context and Approach

The dataset contains 5,000 employee records and 39 variables spanning demographics, employment, compensation, performance, learning and engagement. The assignment emphasizes **reasoning and justification**, so each major preprocessing decision is tied to a data-quality observation or business interpretation.

The workflow is intentionally leakage-safe: the target is separated first, a stratified train/test split is created, and the preprocessing pipeline is fitted only on the training data.


In [0]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, FunctionTransformer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

In [0]:
DATA_PATH = '/Workspace/Users/kushagrakumar1205@gmail.com/bits_msc_ds_ai/Sem_1/Feature Engineering/Graded Assignment Dataset.csv'

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print(df.head(5).to_string())

## Part A – Understanding the Dataset

The first checks cover dimensions, feature types, target balance, missingness, distributions, outliers and correlations.

In [0]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nTarget distribution:")
print(df["Attrition"].value_counts().rename(index={0:"Stayed (0)",1:"Left (1)"}))
print()
print((df["Attrition"].value_counts(normalize=True)*100).round(2).rename(index={0:"Stayed (0)",1:"Left (1)"}).to_frame("Percent"))

type_summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique_values": df.nunique()
})
print(type_summary)


In [0]:
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print(missing.to_frame("Missing_Count"))

plt.figure(figsize=(8,4.5))
missing.sort_values().plot(kind="barh")
plt.title("Missing Values by Feature")
plt.xlabel("Number of missing observations")
plt.tight_layout()
plt.show()


### Missing-value observation

Only four variables contain missing values: **Training_Hours, Annual_Bonus, Manager_Rating and Environment_Satisfaction**. Their missingness is relatively limited (5–7% of rows), making median imputation for numerical variables a simple and robust baseline. No row deletion is used because removing otherwise useful employee records could unnecessarily reduce the sample.

For a categorical variable, the pipeline uses the most-frequent category. The imputation statistics are learned from the training data only.


In [0]:
numeric_predictors = df.select_dtypes(include=np.number).columns.drop("Attrition").tolist()
skewness = df[numeric_predictors].skew().sort_values(ascending=False)
print(skewness.head(15).to_frame("Skewness"))

selected_dist = [
    "Monthly_Salary", "Annual_Bonus", "Stock_Option_Value",
    "Overtime_Hours", "Distance_From_Home", "Years_in_Current_Role"
]
fig, axes = plt.subplots(2,3, figsize=(12,7))
for ax, c in zip(axes.ravel(), selected_dist):
    ax.hist(df[c].dropna(), bins=30)
    ax.set_title(c)
    ax.set_xlabel(c)
    ax.set_ylabel("Frequency")
plt.suptitle("Selected Numerical Distributions", y=1.02)
plt.tight_layout()
plt.show()


In [0]:
outlier_rows = []
for c in numeric_predictors:
    s = df[c].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    n = int(((s < lower) | (s > upper)).sum())
    outlier_rows.append([c, n, round(100*n/len(s),2)])

outlier_df = pd.DataFrame(
    outlier_rows,
    columns=["Feature","IQR_Outliers","Outlier_Percent"]
).sort_values("IQR_Outliers", ascending=False)
print(outlier_df.head(15))


### Distribution and outlier decisions

Several monetary and workload variables are strongly right-skewed, particularly **Stock_Option_Value, Annual_Bonus, Annual_Compensation, Monthly_Salary, Distance_From_Home and Overtime_Hours**. IQR analysis also identifies genuine extreme observations.

Rather than deleting or aggressively clipping observations, the pipeline:
1. applies **log1p** to selected positive, right-skewed variables;
2. uses **RobustScaler**, whose median/IQR scaling is less sensitive to extreme values.

This preserves potentially meaningful high-compensation or high-workload employees instead of assuming every extreme value is an error.


In [0]:
numeric_for_corr = df.select_dtypes(include=np.number).drop(columns="Attrition")
corr = numeric_for_corr.corr().abs()

pairs = []
for i, c1 in enumerate(corr.columns):
    for j, c2 in enumerate(corr.columns[:i]):
        if corr.loc[c1,c2] >= 0.75:
            pairs.append([c1,c2,corr.loc[c1,c2]])
high_corr_df = pd.DataFrame(
    pairs, columns=["Feature_1","Feature_2","Absolute_Correlation"]
).sort_values("Absolute_Correlation", ascending=False)

print(high_corr_df)

plt.figure(figsize=(12,10))
plt.imshow(numeric_for_corr.corr(), aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(numeric_for_corr.columns)), numeric_for_corr.columns, rotation=90, fontsize=6)
plt.yticks(range(len(numeric_for_corr.columns)), numeric_for_corr.columns, fontsize=6)
plt.title("Numerical Feature Correlation Matrix")
plt.tight_layout()
plt.show()


### Key correlation findings

Important high-correlation pairs include:

- **Age ↔ Total_Work_Experience (~0.94):** both represent career maturity; total experience is more directly tied to employment history.
- **Annual_Compensation ↔ Stock_Option_Value (~0.92):** compensation is partly composed of stock value.
- **Monthly_Salary ↔ Monthly_Expenses (~0.91):** expenses scale strongly with salary.
- **Productivity_Index ↔ Performance_Rating (~0.88):** overlapping performance information.
- **Manager_Rating ↔ Performance_Rating (~0.86):** related performance assessments.
- **Promotion_Count ↔ Years_at_Company (~0.79):** promotion opportunity is naturally related to tenure.

The threshold is used as a diagnostic rather than an automatic deletion rule. Business meaning is considered before removing variables.


In [0]:
for c in ["Department","Job_Role","Work_Mode","Compensation_Band","Experience_Band"]:
    rates = df.groupby(c)["Attrition"].mean().sort_values()
    print(f"\n{c}")
    print(rates.to_frame("Attrition_Rate"))

fig, axes = plt.subplots(1,3, figsize=(14,4.5))
for ax, c in zip(axes, ["Department","Job_Role","Work_Mode"]):
    rates = df.groupby(c)["Attrition"].mean().sort_values()
    ax.barh(rates.index, rates.values)
    ax.set_title(f"Attrition Rate by {c}")
    ax.set_xlabel("Attrition rate")
plt.tight_layout()
plt.show()


## Part B – Feature Engineering Strategy

### 1. Identifier handling
**Employee_ID is removed** because it is a unique identifier and has no meaningful generalizable business interpretation.

### 2. Ordinal handling
**Education_Level** is mapped to an ordered numeric representation:

`High School = 1 < Bachelor = 2 < Master = 3 < Doctorate = 4`

This retains the natural order instead of treating education as unrelated nominal categories.

### 3. Categorical encoding
The nominal variables **Gender, Marital_Status, Department, Job_Role, Business_Travel and Work_Mode** are one-hot encoded. `handle_unknown="ignore"` makes the pipeline safer when a future dataset contains a previously unseen category.

### 4. Missing values
Numerical values use **median imputation**; categorical values use **most-frequent imputation**.

### 5. Skew and outliers
Selected right-skewed positive variables receive **log1p**, followed by **RobustScaler**. Other numerical variables receive median imputation and RobustScaler.

### 6. Why no blanket standardization?
Robust scaling is preferred because the dataset intentionally contains outliers. The objective is not to erase legitimate extreme employees but to reduce their influence on downstream algorithms.


## Part C – Feature Construction and Feature Selection

The following constructed variables translate raw HR measurements into more interpretable ratios or composite indicators.

In [0]:
class HRFeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        education_map = {
            "High School": 1, "Bachelor": 2,
            "Master": 3, "Doctorate": 4
        }
        X["Education_Level_Ordinal"] = X["Education_Level"].map(education_map)

        X["Tenure_to_Experience_Ratio"] = X["Years_at_Company"] / (X["Total_Work_Experience"] + 1)
        X["Role_Tenure_Ratio"] = X["Years_in_Current_Role"] / (X["Years_at_Company"] + 1)
        X["Promotion_Rate"] = X["Promotion_Count"] / (X["Years_at_Company"] + 1)
        X["Overtime_Burden"] = X["Overtime_Hours"] / 160.0
        X["Expense_to_Salary_Ratio"] = X["Monthly_Expenses"] / X["Monthly_Salary"]
        X["Bonus_Rate"] = X["Annual_Bonus"] / (X["Monthly_Salary"] * 12)

        satisfaction_cols = [
            "Job_Satisfaction","Work_Life_Balance",
            "Environment_Satisfaction","Relationship_Satisfaction",
            "Recognition_Score","Manager_Support"
        ]
        X["Satisfaction_Index"] = X[satisfaction_cols].mean(axis=1)
        X["Career_Mobility_Index"] = (
            X["Promotion_Count"] + X["Career_Growth_Index"]
        ) / 2.0

        drop_cols = [
            "Employee_ID","Age","Annual_Compensation",
            "Experience_Band","Compensation_Band",
            "Monthly_Expenses","Productivity_Index",
            "Education_Level"
        ]
        return X.drop(columns=drop_cols, errors="ignore")

engineer = HRFeatureEngineer()
X_engineered = engineer.fit_transform(df.drop(columns="Attrition"))
print("Columns after feature construction/selection:", X_engineered.shape[1])
print(X_engineered.head())


### Engineered features and business intuition

| Engineered feature | Business intuition |
|---|---|
| `Tenure_to_Experience_Ratio` | Measures how much of an employee's career has been spent at the current organization. |
| `Role_Tenure_Ratio` | Measures how concentrated company tenure is in the current role; low values can indicate movement across roles. |
| `Promotion_Rate` | Normalizes promotions by tenure, making employees with different company tenures more comparable. |
| `Overtime_Burden` | Converts overtime into a workload burden relative to an approximate 160-hour monthly baseline. |
| `Expense_to_Salary_Ratio` | Captures financial burden relative to monthly salary rather than using expenses alone. |
| `Bonus_Rate` | Normalizes annual bonus by annual base salary. |
| `Satisfaction_Index` | Combines six related employee-experience ratings into one interpretable overall satisfaction signal. |
| `Career_Mobility_Index` | Combines promotion history and career-growth information into a broader mobility indicator. |

The `+1` denominator adjustment in tenure ratios prevents division by zero for employees with zero tenure/experience. Missing inputs are not manually filled here; the downstream imputation step handles them consistently.


### Feature selection decisions

The following variables are removed before preprocessing:

- `Employee_ID`: identifier, not a predictive employee characteristic.
- `Age`: highly redundant with `Total_Work_Experience`; total experience is more directly relevant to career history.
- `Annual_Compensation`: redundant with salary/bonus/stock components and therefore risks duplicating compensation information.
- `Compensation_Band`: coarse version of compensation; continuous salary information is retained.
- `Experience_Band`: coarse version of total experience; continuous experience is retained.
- `Monthly_Expenses`: strongly correlated with salary; the normalized `Expense_to_Salary_Ratio` is retained.
- `Productivity_Index`: highly correlated with `Performance_Rating`; performance rating is retained as the simpler direct performance measure.

This is a **business-informed selection**, not a blind correlation filter. Moderately correlated variables such as `Engagement_Index` and `Attendance_Score` are retained because they measure different HR concepts.


In [0]:
engineered_names = [
    "Tenure_to_Experience_Ratio","Role_Tenure_Ratio",
    "Promotion_Rate","Overtime_Burden",
    "Expense_to_Salary_Ratio","Satisfaction_Index"
]

temp = df.copy()
temp["Tenure_to_Experience_Ratio"] = temp["Years_at_Company"]/(temp["Total_Work_Experience"]+1)
temp["Role_Tenure_Ratio"] = temp["Years_in_Current_Role"]/(temp["Years_at_Company"]+1)
temp["Promotion_Rate"] = temp["Promotion_Count"]/(temp["Years_at_Company"]+1)
temp["Overtime_Burden"] = temp["Overtime_Hours"]/160
temp["Expense_to_Salary_Ratio"] = temp["Monthly_Expenses"]/temp["Monthly_Salary"]
temp["Satisfaction_Index"] = temp[
    ["Job_Satisfaction","Work_Life_Balance","Environment_Satisfaction",
     "Relationship_Satisfaction","Recognition_Score","Manager_Support"]
].mean(axis=1)

print(temp.groupby("Attrition")[engineered_names].mean().round(3))

means = temp.groupby("Attrition")[engineered_names].mean().T
means.plot(kind="bar", figsize=(10,6))
plt.title("Mean Engineered Feature Values by Attrition Class")
plt.ylabel("Mean value")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Part D – Dimensionality Reduction and Reusable Pipeline

One-hot encoding expands the feature space. After engineering and encoding, the pipeline produces a wider numeric matrix than the original raw feature table. PCA is therefore evaluated as an **optional downstream dimensionality-reduction step** rather than being used automatically.

A 95% cumulative explained-variance criterion is used. This keeps most of the transformed variance while reducing the number of dimensions.


In [0]:
cat_cols = [
    "Gender","Marital_Status","Department",
    "Job_Role","Business_Travel","Work_Mode"
]

log_cols = [
    "Distance_From_Home","Monthly_Salary","Annual_Bonus",
    "Stock_Option_Value","Overtime_Hours",
    "Years_in_Current_Role","Certification_Count",
    "Years_at_Company","Promotion_Count",
    "Bonus_Rate","Expense_to_Salary_Ratio"
]

all_num = [c for c in X_engineered.columns if c not in cat_cols]
log_cols = [c for c in log_cols if c in all_num]
robust_cols = [c for c in all_num if c not in log_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("log_numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
            ("scaler", RobustScaler())
        ]), log_cols),

        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler())
        ]), robust_cols),

        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_cols)
    ],
    remainder="drop"
)

preprocessing_pipeline = Pipeline([
    ("feature_engineering", HRFeatureEngineer()),
    ("preprocessing", preprocessor)
])

X = df.drop(columns="Attrition")
y = df["Attrition"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

X_train_processed = preprocessing_pipeline.fit_transform(X_train)
X_test_processed = preprocessing_pipeline.transform(X_test)

print("Raw training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)


In [0]:
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train_processed)

print("PCA components for 95% explained variance:", pca.n_components_)
print("Original processed dimensions:", X_train_processed.shape[1])
print("Reduced dimensions:", X_train_pca.shape[1])
print("Explained variance retained:", round(pca.explained_variance_ratio_.sum(), 4))

cum_var = np.cumsum(pca.explained_variance_ratio_)
plt.figure(figsize=(8,4.5))
plt.plot(range(1, len(cum_var)+1), cum_var, marker="o", markersize=3)
plt.axhline(0.95, linestyle="--")
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA Explained Variance")
plt.tight_layout()
plt.show()


### PCA conclusion

The processed training matrix contains **58 dimensions**, while PCA needs **31 components** to retain at least 95% of variance. This is approximately a **46.6% reduction in dimensionality**.

PCA is therefore potentially useful when a downstream model is sensitive to dimensionality, multicollinearity or computational cost. However, it is **not made mandatory in the core preprocessing pipeline**, because PCA reduces interpretability: transformed components no longer correspond directly to HR variables.

Recommended practice:
- Use the engineered, interpretable pipeline as the default.
- Add PCA as an optional downstream step for algorithms or experiments where dimensionality reduction provides a measurable benefit.


## Final Pipeline Summary

**Raw HR data → target separation → feature construction/selection → missing-value imputation → log transformation of selected skewed variables → robust scaling → one-hot encoding of nominal categories → optional PCA**

The pipeline is reusable because all learned preprocessing statistics (medians, category levels and scaling parameters) are fitted through scikit-learn's pipeline on training data and then applied consistently to future data.


In [0]:
# Final sanity checks
print("Any NaN after preprocessing?", np.isnan(X_train_processed).any())
print("Any infinite values after preprocessing?", np.isinf(X_train_processed).any())
print("Pipeline steps:")
print(preprocessing_pipeline)


## Part E – Summary and Reflection

### Key decisions
- Removed the unique employee identifier.
- Preserved ordinal education information through ordered encoding.
- Used median/mode imputation rather than deleting rows.
- Applied log1p to selected skewed positive variables and RobustScaler to reduce outlier influence.
- One-hot encoded nominal categories with unknown-category protection.
- Constructed tenure, mobility, overtime, financial-burden and satisfaction features.
- Removed several redundant/coarsened variables using both correlation evidence and business meaning.
- Evaluated PCA rather than assuming it was automatically beneficial.

### Challenges
The dataset intentionally combines missing values, skewed distributions, outliers and correlated variables. The main challenge is balancing information preservation against redundancy and interpretability.

### Benefits
The resulting pipeline is reproducible, leakage-safe and suitable for downstream machine-learning workflows. It also separates feature construction from preprocessing, making the logic easier to audit and modify.

### Assumptions and limitations
- `160` hours is used only as an approximate monthly workload baseline for `Overtime_Burden`.
- Ratio features use `+1` in tenure denominators to avoid division by zero.
- Missing numerical values are assumed to be missing-at-random enough for median imputation to be a reasonable baseline.
- Correlation is treated as evidence of redundancy, not proof that one variable is useless.
- No model accuracy claim is made because predictive modelling is outside the assignment requirements.
